# Bagian B — Line Plot & Area Plot

**PJ:** Arfi (Anggota B)

Sel setup bersama sudah terpasang di bawah — jangan diubah. Simpan gambar ke `../figures/01_*.png` dan `../figures/02_*.png`.

## Setup Bersama

> ⚠️ **Sel di bawah ini adalah SETUP BERSAMA (dibuat Bagian A).** Jangan diubah di notebook masing-masing. Kalau perlu perubahan, minta Aprilianza (Anggota A) mengubahnya di `kerja/A_setup_data.ipynb`, lalu semua anggota copy ulang.

Variabel yang tersedia setelah sel ini dijalankan:

| Variabel | Isi |
|---|---|
| `TICKERS` | `['BBCA', 'BBRI', 'BMRI', 'BBNI']` (urutan tetap) |
| `NAMA_BANK` | nama lengkap tiap bank |
| `WARNA` | warna tetap per bank (selalu pakai ini, jangan pilih warna sendiri) |
| `df_all` | long format: `Date, Ticker, Open, High, Low, Close, Adj Close, Volume` |
| `adj_wide` | pivot **Adj Close** (index `Date`, kolom = ticker) |
| `vol_wide` | pivot **Volume** (index `Date`, kolom = ticker) |
| `HARI_DIBUANG` | tanggal placeholder hari libur (Volume 0, harga datar) yang dibuang |
| `FIG_DIR` | folder `figures/` untuk menyimpan gambar |
| `simpan_fig(fig, nama)` | simpan gambar ke `figures/<nama>.png` |
| `format_tanggal(ax)` | rapikan sumbu-x tanggal (label per tahun) |

Notebook bisa dijalankan dari root repo, dari folder `kerja/`, atau dari Google Colab (data otomatis diambil dari GitHub).

In [ ]:
# ===== SETUP BERSAMA (Bagian A) - JANGAN DIUBAH, copy persis ke semua notebook =====
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

TICKERS = ['BBCA', 'BBRI', 'BMRI', 'BBNI']
NAMA_BANK = {
    'BBCA': 'Bank Central Asia',
    'BBRI': 'Bank Rakyat Indonesia',
    'BMRI': 'Bank Mandiri',
    'BBNI': 'Bank Negara Indonesia',
}
# Palet kategorikal tervalidasi (aman untuk buta warna pada grafik garis/stacked).
# Warna mengikuti bank, bukan urutan ranking - jangan ditukar antar plot.
WARNA = {
    'BBCA': '#2a78d6',  # biru
    'BBRI': '#eb6834',  # oranye
    'BMRI': '#1baf7a',  # aqua
    'BBNI': '#eda100',  # kuning
}

SUMBER_DATA = 'https://www.kaggle.com/datasets/cmglonly/dataset-saham-bank-indonesia-2014-2024/data'
GITHUB_RAW = 'https://raw.githubusercontent.com/aprilianza/Midterm_DataVisualization/main'

# Cari root project: bisa dijalankan dari root repo maupun dari folder kerja/.
# Jika tidak ketemu (mis. di Colab), data diambil langsung dari GitHub.
ROOT = next((p for p in [Path.cwd(), Path.cwd().parent] if (p / 'dataset').is_dir()), None)
FIG_DIR = (ROOT or Path.cwd()) / 'figures'
FIG_DIR.mkdir(exist_ok=True)


def load_saham(ticker):
    nama_file = f'{ticker}.JK_10_years.csv'
    sumber = ROOT / 'dataset' / nama_file if ROOT else f'{GITHUB_RAW}/dataset/{nama_file}'
    df = pd.read_csv(sumber)
    # Kolom Date bertimezone UTC (00:00+00:00) -> jadikan tanggal biasa tanpa timezone
    df['Date'] = pd.to_datetime(df['Date'], utc=True).dt.tz_localize(None).dt.normalize()
    df['Ticker'] = ticker
    return df.sort_values('Date')


df_all = (
    pd.concat([load_saham(t) for t in TICKERS], ignore_index=True)
    .drop_duplicates(subset=['Ticker', 'Date'])
    [['Date', 'Ticker', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']]
    .reset_index(drop=True)
)
# Baris placeholder hari libur bursa (Volume 0 & harga datar O=H=L=C) dari Yahoo Finance
# dibuang untuk SEMUA bank agar tanggal di adj_wide/vol_wide tetap sejajar.
_placeholder = (df_all['Volume'] == 0) & df_all[['Open', 'High', 'Low']].eq(df_all['Close'], axis=0).all(axis=1)
HARI_DIBUANG = pd.DatetimeIndex(sorted(df_all.loc[_placeholder, 'Date'].unique()))
df_all = df_all[~df_all['Date'].isin(HARI_DIBUANG)].reset_index(drop=True)

adj_wide = df_all.pivot(index='Date', columns='Ticker', values='Adj Close')[TICKERS]
vol_wide = df_all.pivot(index='Date', columns='Ticker', values='Volume')[TICKERS]

# Style plot seragam untuk semua anggota
plt.rcParams.update({
    'figure.figsize': (12, 5),
    'figure.dpi': 110,
    'savefig.dpi': 200,
    'savefig.bbox': 'tight',
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
    'axes.titlelocation': 'left',
    'axes.labelcolor': '#52514e',
    'axes.edgecolor': '#b5b4ae',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'axes.axisbelow': True,
    'grid.color': '#e6e5e0',
    'grid.linewidth': 0.8,
    'xtick.color': '#52514e',
    'ytick.color': '#52514e',
    'lines.linewidth': 1.6,
    'legend.frameon': False,
})


def simpan_fig(fig, nama):
    """Simpan figure ke figures/<nama>.png, mis. simpan_fig(fig, '01_line_bbca')."""
    path = FIG_DIR / f'{nama}.png'
    fig.savefig(path)
    print(f'Tersimpan: {path}')


def format_tanggal(ax):
    """Sumbu-x tanggal dengan label per tahun."""
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))


print(f'Data dimuat dari: {ROOT / "dataset" if ROOT else GITHUB_RAW}')
print(f'df_all: {df_all.shape[0]:,} baris | {df_all["Date"].min():%d %b %Y} - {df_all["Date"].max():%d %b %Y}')
print(f'adj_wide & vol_wide: {adj_wide.shape[0]:,} tanggal x {adj_wide.shape[1]} bank')
print(f'Hari placeholder (libur, Volume 0) yang dibuang: {len(HARI_DIBUANG)} tanggal')

## Bagian B — Line Plot & Area Plot

_(mulai kerjakan di bawah)_